# Import Results

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

path_og = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original"
path_mod_1 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified_v1/"
path_mod_2 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-2/"
path_mod_3 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-3/"
path_mod_4 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-4/"
path_mod_5 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-5/"
path_mod_5_1 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-5_1/"
path_mod_6 = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-6/"

### Lade-Netzwerk-Funktion

In [0]:
def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))

# SR27 Szenario 3, abgeleitete Jahresmengen in TWh
ZIELE_TWH = {
    2040: {"H2 for industry": 86, "gas for industry": 137},
    2045: {"H2 for industry": 107, "gas for industry": 107},
}
CARRIER = ["H2 for industry", "gas for industry"]

## Lade-Industrie-Funktion

In [0]:
def industrie_bedarf(n, year):
    """Industrielle H2-/CH4-Nachfrage in DE (TWh/a) mit Zielabgleich."""
    # Stunden pro Snapshot (z. B. 3 bei 3H-Auflösung)
    w = n.snapshot_weightings.generators
    print(f"Jahr {year}: {len(n.snapshots)} Snapshots, "
          f"Summe Gewichtung = {w.sum():.0f} h")

    loads = n.loads[
        n.loads.carrier.isin(CARRIER)
        & n.loads.bus.str.startswith("DE", na=False)
    ]

    # Prüfen, ob es Zeitreihen gibt (sonst gilt das statische p_set)
    zr = loads.index.intersection(n.loads_t.p_set.columns)
    if len(zr):
        print(f"Achtung: {len(zr)} Lasten haben Zeitreihen in loads_t.p_set")

    # Jahresenergie: tatsächlicher Lastverlauf × Gewichtung
    energie = (
        n.loads_t.p[loads.index]
        .multiply(w, axis=0)
        .sum()
        .groupby(loads.carrier)
        .sum() / 1e6
    )

    df = pd.DataFrame({
        "Anzahl Lasten": loads.groupby("carrier").size(),
        "Basis TWh/a": energie,
        "Mittlere Leistung GW": energie * 1e3 / w.sum(),
        "Ziel TWh/a": pd.Series(ZIELE_TWH[year]),
    })
    df["Faktor (Ziel/Basis)"] = df["Ziel TWh/a"] / df["Basis TWh/a"]
    df["Differenz TWh/a"] = df["Ziel TWh/a"] - df["Basis TWh/a"]
    return df

### Ein Netzwerk prüfen

In [0]:
# # Ein Netzwerk prüfen
# year = 2040
# n = laden(path_og, year)
# display(industrie_bedarf(n, year).round(2))

### Mod-4, Mod-5: Vergleich Industriebedarfe

In [0]:
runs = {"OG": path_og, "Mod 4": path_mod_4, "Mod 5": path_mod_5}

tabellen = []
for lauf, path in runs.items():
    for year in [2040, 2045]:
        df = industrie_bedarf(laden(path, year), year)
        df.insert(0, "Jahr", year)
        df.insert(0, "Lauf", lauf)
        tabellen.append(df)

display(pd.concat(tabellen).round(2))

### Test: Sind wir on track?

In [0]:
gas = ["OCGT", "CCGT", "urban central gas CHP", "urban central gas CHP CC"]
h2 = ["H2 OCGT", "H2 CCGT", "urban central H2 CHP",
      "H2 retrofit OCGT", "H2 retrofit CCGT", "urban central H2 retrofit CHP"]
ziele = {2040: {"Methan": 50, "H2": 10}, 2045: {"Methan": 19, "H2": 41}}


def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))


def leistung(n, carrier, fest=False):
    """Elektrische Leistung deutscher Kraftwerke in GW_el."""
    l = n.links
    mask = (
        l.carrier.isin(carrier)
        & l.bus1.str.startswith("DE", na=False)
        & l.bus1.map(n.buses.carrier).eq("AC")
        & l.active
    )
    if fest:
        mask &= ~l.p_nom_extendable
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    return (p[mask] * l.loc[mask, "efficiency"]).sum() / 1e3


def elektrolyse(n):
    """Elektrolyseleistung in DE in GW_el (Eingangsseite)."""
    l = n.links
    mask = (
        l.carrier.eq("H2 Electrolysis")
        & l.bus0.str.startswith("DE", na=False)
        & l.active
    )
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    return p[mask].sum() / 1e3


def gasbilanz(n):
    """Energiebilanz aller deutschen Gasbusse je Carrier in TWh."""
    bilanz = n.statistics.energy_balance(groupby=["bus", "carrier"])
    bilanz = bilanz.droplevel("component")
    bus = bilanz.index.get_level_values("bus")
    de = bilanz[bus.str.startswith("DE") & bus.str.contains("gas")]
    return de.groupby(level="carrier").sum() / 1e6


# Netze einmal laden
nets = {
    (lauf, year): laden(path, year)
    for lauf, path in [("Mod 4", path_mod_4), ("Mod 5", path_mod_5)]
    for year in [2040, 2045]
}

# 1. Kraftwerksziele und Systemkosten
rows = []
for (lauf, year), n in nets.items():
    rows.append({
        "Lauf": lauf,
        "Jahr": year,
        "Methan GW_el": leistung(n, gas),
        "Ziel Methan": ziele[year]["Methan"],
        "H2 GW_el": leistung(n, h2),
        "Ziel H2": ziele[year]["H2"],
        "Elektrolyse GW_el": elektrolyse(n),
        "Zielfunktion Mrd. EUR": n.objective / 1e9,
    })
display(pd.DataFrame(rows).sort_values(["Jahr", "Lauf"]).round(2))

# 2. Gasbilanz DE: Woher kommt das zusätzliche Methan?
for year in [2040, 2045]:
    vergleich = pd.DataFrame({
        "Mod 4": gasbilanz(nets[("Mod 4", year)]),
        "Mod 5": gasbilanz(nets[("Mod 5", year)]),
    }).fillna(0)
    vergleich["Differenz"] = vergleich["Mod 5"] - vergleich["Mod 4"]
    vergleich = vergleich[vergleich.abs().max(axis=1) > 0.1]
    print(f"--- Gasbilanz DE {year} (TWh, + Einspeisung / − Entnahme) ---")
    print(vergleich.sort_values("Differenz").round(2).to_string())

Ergebnis: 
- Die Kraftwerksziele sind bestätigt: Methan, H₂ und Elektrolyse treffen in Mod 5 in beiden Jahren exakt 50/10/17 bzw. 19/41/17 GW_el. Die höhere Industrienachfrage stört Mod-1 bis Mod-4 also nicht.


### Mod-5: Biogas 

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))

for year in [2040, 2045]:
    n = laden(path_mod_5, year)
    w = n.snapshot_weightings.generators
    print(f"=============== {year} ===============")

    # 1. Biogas-Potenzial und tatsächliche Nutzung in DE
    g = n.generators[
        n.generators.carrier.str.contains("biogas")
        & n.generators.bus.str.startswith("DE")
    ]
    erzeugung = (
        n.generators_t.p.reindex(columns=g.index, fill_value=0)
        .multiply(w, axis=0).sum() / 1e6
    )
    print(pd.DataFrame({
        "Potenzial TWh": g.e_sum_max / 1e6,
        "Genutzt TWh": erzeugung,
        "Grenzkosten EUR/MWh": g.marginal_cost,
    }).groupby(g.carrier).agg(
        {"Potenzial TWh": "sum", "Genutzt TWh": "sum", "Grenzkosten EUR/MWh": "mean"}
    ).round(2).to_string())

    # 2. Aufbereitung zu Biomethan
    l = n.links[
        n.links.carrier.str.startswith("biogas to gas")
        & n.links.bus0.str.startswith("DE")
    ]
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    p0 = n.links_t.p0.reindex(columns=l.index, fill_value=0)
    biomethan = p0.multiply(w, axis=0).sum() * l.efficiency / 1e6
    print(pd.DataFrame({
        "Leistung GW": p / 1e3,
        "Wirkungsgrad": l.efficiency,
        "Biomethan TWh": biomethan,
    }).groupby(l.carrier).agg(
        {"Leistung GW": "sum", "Wirkungsgrad": "mean", "Biomethan TWh": "sum"}
    ).round(2).to_string())

    # 3. CO2-Preis: bindet die Emissionsgrenze?
    gc = n.global_constraints
    print(gc[gc.index.str.contains("co2", case=False)][["constant", "mu"]].round(2).to_string())

### Mod-5.1: Tabellenwerte prüfen

In [0]:
laeufe = {
    "Mod 5": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-5/",
    "Mod 5 neu": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-6/",
}
gas = ["OCGT", "CCGT", "urban central gas CHP", "urban central gas CHP CC"]
h2 = ["H2 OCGT", "H2 CCGT", "urban central H2 CHP",
      "H2 retrofit OCGT", "H2 retrofit CCGT", "urban central H2 retrofit CHP"]


def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))


def leistung(n, carrier):
    l = n.links
    mask = (l.carrier.isin(carrier) & l.bus1.str.startswith("DE", na=False)
            & l.bus1.map(n.buses.carrier).eq("AC") & l.active)
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    return (p[mask] * l.loc[mask, "efficiency"]).sum() / 1e3


def last_twh(n, carrier):
    w = n.snapshot_weightings.generators
    i = n.loads.index[n.loads.carrier.eq(carrier) & n.loads.bus.str.startswith("DE")]
    return n.loads_t.p[i].multiply(w, axis=0).sum().sum() / 1e6


def gas_twh(n, carrier):
    bilanz = n.statistics.energy_balance(groupby=["bus", "carrier"]).droplevel("component")
    bus = bilanz.index.get_level_values("bus")
    de = bilanz[bus.str.startswith("DE") & bus.str.contains("gas")]
    return de.groupby(level="carrier").sum().get(carrier, 0) / 1e6


rows = []
for lauf, path in laeufe.items():
    for year in [2040, 2045]:
        n = laden(path, year)
        co2 = n.global_constraints.mu.get("CO2Limit", float("nan"))
        rows.append({
            "Lauf": lauf, "Jahr": year,
            "H2 Ind. TWh": last_twh(n, "H2 for industry"),
            "CH4 Ind. TWh": last_twh(n, "gas for industry"),
            "Methan-KW GW_el": leistung(n, gas),
            "H2-KW GW_el": leistung(n, h2),
            "Erdgas fossil TWh": gas_twh(n, "gas primary"),
            "Biogas TWh": gas_twh(n, "biogas"),
            "CO2-Preis EUR/t": -co2,
            "Zielfunktion Mrd. EUR": n.objective / 1e9,
        })

print(pd.DataFrame(rows).sort_values(["Jahr", "Lauf"]).round(2).to_string(index=False))

**Alles wie erwartet, der Lauf ist plausibel.**

### Mod-6:
**Prüfen von:**
- Vorgaben eingehalten: Biomethan genau 70 TWh, deutsches Biogas-Potenzial bei 107 TWh, Industrie- und Kraftwerksziele weiter erfüllt.
- Woher kommt das Biomethan: wie viel Aufbereitung das Modell baut, mit oder ohne CC, und was aus den Anlagen von 2040 wird.
- Was es verdrängt: Erwartet ist ein Rückgang von fossilem Erdgas um rund 40 TWh (2040) bzw. 70 TWh (2045).
- Was es kostet: Zielfunktion und CO₂-Preis im Vergleich zu Mod-5_1.

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

laeufe = {
    "Mod 5_1": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-5_1/",
    "Mod 6": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-6/",
}
gas = ["OCGT", "CCGT", "urban central gas CHP", "urban central gas CHP CC"]
h2 = ["H2 OCGT", "H2 CCGT", "urban central H2 CHP",
      "H2 retrofit OCGT", "H2 retrofit CCGT", "urban central H2 retrofit CHP"]


def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))


def leistung(n, carrier):
    l = n.links
    mask = (l.carrier.isin(carrier) & l.bus1.str.startswith("DE", na=False)
            & l.bus1.map(n.buses.carrier).eq("AC") & l.active)
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)
    return (p[mask] * l.loc[mask, "efficiency"]).sum() / 1e3


def last_twh(n, carrier):
    w = n.snapshot_weightings.generators
    i = n.loads.index[n.loads.carrier.eq(carrier) & n.loads.bus.str.startswith("DE")]
    return n.loads_t.p[i].multiply(w, axis=0).sum().sum() / 1e6


def gas_twh(n, carrier):
    bilanz = n.statistics.energy_balance(groupby=["bus", "carrier"]).droplevel("component")
    bus = bilanz.index.get_level_values("bus")
    de = bilanz[bus.str.startswith("DE") & bus.str.contains("gas")]
    return de.groupby(level="carrier").sum().get(carrier, 0) / 1e6


def biogas_details(n):
    """Potenzial, Biomethan je Aufbereitungsart und Leistung (fest/neu) in DE."""
    w = n.snapshot_weightings.generators
    g = n.generators[n.generators.carrier.eq("biogas") & n.generators.bus.str.startswith("DE")]
    l = n.links[n.links.carrier.isin(["biogas to gas", "biogas to gas CC"])
                & n.links.bus0.str.startswith("DE")]
    p0 = n.links_t.p0.reindex(columns=l.index, fill_value=0)
    out = p0.multiply(w, axis=0).sum() * l.efficiency / 1e6
    p = l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt) / 1e3
    return {
        "Potenzial TWh": g.e_sum_max.sum() / 1e6,
        "Biomethan TWh": out.sum(),
        "davon mit CC TWh": out[l.carrier.eq("biogas to gas CC")].sum(),
        "Aufbereitung Bestand GW": p[~l.p_nom_extendable].sum(),
        "Aufbereitung Neubau GW": p[l.p_nom_extendable].sum(),
    }


rows = []
for lauf, path in laeufe.items():
    for year in [2040, 2045]:
        n = laden(path, year)
        rows.append({
            "Lauf": lauf, "Jahr": year,
            "H2 Ind. TWh": last_twh(n, "H2 for industry"),
            "CH4 Ind. TWh": last_twh(n, "gas for industry"),
            "Methan-KW GW_el": leistung(n, gas),
            "H2-KW GW_el": leistung(n, h2),
            **biogas_details(n),
            "Erdgas fossil TWh": gas_twh(n, "gas primary"),
            "CO2-Preis EUR/t": -n.global_constraints.mu.get("CO2Limit", float("nan")),
            "Zielfunktion Mrd. EUR": n.objective / 1e9,
        })

df = pd.DataFrame(rows).sort_values(["Jahr", "Lauf"]).round(2)
print(df.T.to_string())  # transponiert, damit alle Spalten lesbar bleiben

**Prep Mod-7: Anpassungen fehlender Mantelzahlen Private Haushalte & GHD**

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

path = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-6/"

def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))

# Zuordnung Modell-Carrier -> SR-Sektor (unbekannte landen unter "Sonstige: <carrier>")
CH4_SEKTOR = {
    "OCGT": "Kraftwerke", "CCGT": "Kraftwerke",
    "urban central gas CHP": "Kraftwerke", "urban central gas CHP CC": "Kraftwerke",
    "gas for industry": "Industrie", "gas for industry CC": "Industrie",
    "rural gas boiler": "Haushalte + GHD", "urban decentral gas boiler": "Haushalte + GHD",
    "urban central gas boiler": "Fernwärme",
    "SMR": "Umwandlung (SMR)", "SMR CC": "Umwandlung (SMR)",
    "gas primary": "Aufkommen fossil", "biogas": "Aufkommen Biogas",
    "renewable gas": "Aufkommen EE-Gas Import/Export", "Sabatier": "Aufkommen Sabatier",
    "gas compressing": "Verluste Verdichtung",
}
H2_SEKTOR = {
    "H2 for industry": "Industrie",
    "H2 OCGT": "Kraftwerke", "H2 CCGT": "Kraftwerke",
    "H2 retrofit OCGT": "Kraftwerke", "H2 retrofit CCGT": "Kraftwerke",
    "urban central H2 CHP": "Kraftwerke", "urban central H2 retrofit CHP": "Kraftwerke",
    "H2 Fuel Cell": "Kraftwerke",
    "land transport fuel cell": "Verkehr",
    "methanolisation": "Umwandlung (E-Fuels)", "Fischer-Tropsch": "Umwandlung (E-Fuels)",
    "Sabatier": "Umwandlung (E-Fuels)", "Haber-Bosch": "Umwandlung (E-Fuels)",
    "H2 Electrolysis": "Aufkommen Elektrolyse",
    "SMR": "Aufkommen SMR", "SMR CC": "Aufkommen SMR",
    "H2 pipeline": "Netto-Import Pipeline", "H2 pipeline (Kernnetz)": "Netto-Import Pipeline",
    "H2 pipeline retrofitted": "Netto-Import Pipeline",
    "H2 Store": "Speicher (netto)",
}
SR_CH4 = {2040: {"Kraftwerke": 0, "Industrie": 130, "Haushalte + GHD": 219, "Fernwärme": 15},
          2045: {"Kraftwerke": 56, "Industrie": 101, "Haushalte + GHD": 76, "Fernwärme": 15}}
SR_H2 = {2040: {"Kraftwerke": 20, "Industrie": 79}, 2045: {"Kraftwerke": 27, "Industrie": 101}}


def sektorbilanz(n, bus_filter, mapping):
    """Energiebilanz deutscher Busse in TWh, nach SR-Sektor (− Entnahme, + Aufkommen)."""
    b = n.statistics.energy_balance(groupby=["bus", "carrier"], nice_names=False)
    b = b.droplevel("component")
    bus = b.index.get_level_values("bus")
    b = b[bus.str.startswith("DE") & bus_filter(bus)]
    b = b.groupby(level="carrier").sum() / 1e6
    sektor = b.index.map(lambda c: mapping.get(c, f"Sonstige: {c}"))
    return b.groupby(sektor).sum()


for year in [2040, 2045]:
    n = laden(path, year)
    ch4 = sektorbilanz(n, lambda bus: bus.str.contains("gas"), CH4_SEKTOR)
    h2 = sektorbilanz(n, lambda bus: bus.map(n.buses.carrier) == "H2", H2_SEKTOR)

    for name, bil, sr in [("Methan", ch4, SR_CH4[year]), ("Wasserstoff", h2, SR_H2[year])]:
        df = pd.DataFrame({"Modell TWh": bil.round(1)})
        df["SR S3 TWh (Entnahme)"] = pd.Series(sr).reindex(df.index)
        df = df[df["Modell TWh"].abs() > 0.1]
        print(f"--- {name} DE {year} (− Entnahme / + Aufkommen) ---")
        print(df.sort_values("Modell TWh").to_string())
        print()

### Mod-7: Gebäude Sektor

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

laeufe = {
    "OG": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original",
    "Mod 6": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-6/",
    "Mod 7": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-7/",
}
GAS_KW = ["OCGT", "CCGT", "urban central gas CHP", "urban central gas CHP CC"]
H2_KW = ["H2 OCGT", "H2 CCGT", "urban central H2 CHP",
         "H2 retrofit OCGT", "H2 retrofit CCGT", "urban central H2 retrofit CHP"]


def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))


def p_nom(l):
    return l.p_nom.where(~l.p_nom_extendable, l.p_nom_opt)


def kw_gw_el(n, carrier):
    l = n.links
    m = (l.carrier.isin(carrier) & l.bus1.str.startswith("DE", na=False)
         & l.bus1.map(n.buses.carrier).eq("AC") & l.active)
    return (p_nom(l)[m] * l.efficiency[m]).sum() / 1e3


def fluss_twh(n, maske, port="p0", eff=False):
    w = n.snapshot_weightings.generators
    i = n.links.index[maske]
    p = getattr(n.links_t, port).reindex(columns=i, fill_value=0).multiply(w, axis=0).sum()
    return (p * n.links.efficiency[i] if eff else p).sum() / 1e6


def last_twh(n, carrier):
    w = n.snapshot_weightings.generators
    i = n.loads.index[n.loads.carrier.eq(carrier) & n.loads.bus.str.startswith("DE")]
    return n.loads_t.p[i].multiply(w, axis=0).sum().sum() / 1e6


def de_emissionen(n):
    """CO2 deutscher Links in die Atmosphäre je Carrier (Mt; + Emission, − Entnahme)."""
    w = n.snapshot_weightings.generators
    l = n.links[n.links.index.str.startswith("DE")]
    em = pd.Series(0.0, index=l.index)
    for port in [c[3:] for c in n.links.columns if c.startswith("bus") and c[3:].isdigit()]:
        idx = l.index[l[f"bus{port}"] == "co2 atmosphere"]
        if len(idx) and f"p{port}" in n.links_t:
            em[idx] += -n.links_t[f"p{port}"].reindex(columns=idx, fill_value=0).multiply(w, axis=0).sum()
    return em.groupby(l.carrier).sum() / 1e6


def de_co2_zufluss(n, bus_carrier):
    b = n.statistics.energy_balance(bus_carrier=bus_carrier, groupby=["bus"]).droplevel("component")
    loc = b.index.map(n.buses.location).fillna("")
    return b[loc.str.startswith("DE") & (b > 0)].sum() / 1e6


rows, emissionen = [], {}
for lauf, path in laeufe.items():
    for year in [2040, 2045]:
        n = laden(path, year)
        l, gc = n.links, n.global_constraints
        de = l.index.str.startswith("DE")
        kessel = de & l.carrier.str.contains("gas boiler") & ~l.carrier.str.contains("urban central")
        biometh = l.carrier.isin(["biogas to gas", "biogas to gas CC"]) & l.bus0.str.startswith("DE")
        elz = l.carrier.eq("H2 Electrolysis") & l.bus0.str.startswith("DE") & l.active
        em = de_emissionen(n)
        emissionen[(lauf, year)] = em
        rows.append({
            "Lauf": lauf, "Jahr": year,
            # Vorgaben Mod-1 bis Mod-7
            "Elektrolyse GW_el (≥17)": p_nom(l)[elz].sum() / 1e3,
            "H2-KW GW_el": kw_gw_el(n, H2_KW),
            "Methan-KW GW_el": kw_gw_el(n, GAS_KW),
            "H2 Industrie TWh": last_twh(n, "H2 for industry"),
            "CH4 Industrie TWh": last_twh(n, "gas for industry"),
            "Biomethan TWh": fluss_twh(n, biometh, eff=True),
            "Gas Gebäude TWh": fluss_twh(n, kessel),
            # CO2 und CCS
            "DE Emissionen netto Mt (Näherung)": em.sum(),
            "DE Budget Mt": gc.constant.get("co2_limit-DE", float("nan")) / 1e6,
            "Schattenpreis DE-Budget EUR/t": -gc.mu.get("co2_limit-DE", float("nan")),
            "CO2-Preis EU EUR/t": -gc.mu.get("CO2Limit", float("nan")),
            "Schattenpreis CCS-Grenze EUR/t": gc.mu.get("co2_sequestration_limit", float("nan")),
            "CO2 abgeschieden DE Mt": de_co2_zufluss(n, "co2 stored"),
            "CO2 gespeichert DE Mt": de_co2_zufluss(n, "co2 sequestered"),
            "Zielfunktion Mrd. EUR": n.objective / 1e9,
        })

print(pd.DataFrame(rows).sort_values(["Jahr", "Lauf"]).round(1).T.to_string())

tab = pd.DataFrame(emissionen).fillna(0).round(1)
print("\nCO2 nach Technologie in DE (Mt, + Emission / − Entnahme)")
print(tab[tab.abs().max(axis=1) > 0.5].sort_values(("Mod 7", 2040)).to_string())

**Kernergebnis: Szenario 3 hält das deutsche Klimaziel, aber teurer**

Das deutsche Budget bindet in keinem Lauf (Schattenpreis 0, auch in Mod 7). Selbst mit 219 TWh Gas in Gebäuden bleibt Deutschland unter dem KSG-Budget, also 9 % von 1990 in 2040 und −5 % in 2045.

Die Spalte „DE Emissionen netto (Näherung)“ darf man dafür nicht mit dem Budget vergleichen. Mod 7 zeigt 2040 dort 94,6 Mt bei einem Budget von 94,7 Mt. Das sieht knapp aus, ist es aber nicht. Meine Zelle zählt den Flugverkehr und die Schifffahrt voll. Das echte Budget zählt nur den Inlandsanteil (Luftverkehr Faktor 0,06, Schifffahrt 0,15), das sind allein rund 37 Mt weniger. Außerdem schreibt es importierte E-Fuels und importiertes EE-Gas als Gutschrift an. Verlässlich ist hier nur der Schattenpreis, und der ist 0.

### Mod-8: CO2-Budget

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

laeufe = {
    # "Mod 7 (Obergrenze)": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-7/",
    "OG (Obergrenze)": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/original/", 
    "Mod 8 (CO2-Preis)": "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-8/",
}
JAHRE = [2035, 2040, 2045]
# Inlandsanteile aus dem Solve-Log ("Adding domestic aviation/shipping emissions")
F_LUFT, F_SCHIFF = 0.05888813249756545, 0.15275394216731217


def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))


def jahressumme(n, links, faktor):
    """Σ Link-p (p0) × Faktor × Gewichtung in t."""
    if len(links) == 0:
        return pd.Series(dtype=float)
    w = n.snapshot_weightings.generators
    p0 = n.links_t.p0.reindex(columns=links, fill_value=0)
    return p0.multiply(w, axis=0).sum() * faktor


def de_budget_emissionen(n):
    """Deutsche CO2-Emissionen in der Abgrenzung von add_national_co2_budgets() (Mt, je Carrier)."""
    l = n.links
    ist_de = pd.Series(l.index.str[:2] == "DE", index=l.index)
    teile = []
    ports = [c[3:] for c in l.columns if c.startswith("bus") and c[3:].isdigit()]
    for port in ports:
        idx = l.index[ist_de & (l[f"bus{port}"] == "co2 atmosphere")
                      & ~l.carrier.str.contains("shipping|aviation")]
        eff = -1.0 if port == "0" else l.loc[idx, "efficiency" if port == "1" else f"efficiency{port}"]
        teile.append(jahressumme(n, idx, eff))
    for carrier, f in [("kerosene for aviation", F_LUFT), ("shipping oil", F_SCHIFF),
                       ("shipping methanol", F_SCHIFF)]:
        idx = l.index[ist_de & (l.carrier == carrier)]
        teile.append(jahressumme(n, idx, l.loc[idx, "efficiency2"] * f))
    meoh = l.at["EU industry methanol", "efficiency2"] if "EU industry methanol" in l.index else 0
    for name, f in [("EU renewable oil -> DE oil", -0.2571), ("DE renewable oil -> EU oil", 0.2571),
                    ("EU methanol -> DE methanol", -meoh), ("DE methanol -> EU methanol", meoh),
                    ("EU renewable gas -> DE gas", -0.198), ("DE renewable gas -> EU gas", 0.198)]:
        if name in l.index:
            teile.append(jahressumme(n, pd.Index([name]), f).rename({name: f"Handel: {name}"}))
    s = pd.concat([t for t in teile if len(t)])
    gruppe = s.index.map(lambda i: i if i.startswith("Handel") else l.at[i, "carrier"])
    return s.groupby(gruppe).sum() / 1e6


rows, sektoren, budget = [], {}, {}
for lauf, path in laeufe.items():
    for year in JAHRE:
        n = laden(path, year)
        gc = n.global_constraints
        preis = -n.stores.at["co2 atmosphere", "marginal_cost"]
        eu_mt = n.stores_t.e["co2 atmosphere"].iloc[-1] / 1e6
        de = de_budget_emissionen(n)
        sektoren[(lauf, year)] = de
        if "co2_limit-DE" in gc.index:
            budget[year] = gc.at["co2_limit-DE", "constant"] / 1e6
        rows.append({
            "Lauf": lauf, "Jahr": year,
            "CO2-Preis vorgegeben EUR/t": preis,
            "Schattenpreis EU-Grenze EUR/t": -gc.mu.get("CO2Limit", float("nan")),
            "DE-Budget aktiv": "co2_limit-DE" in gc.index,
            "EU Emissionen netto Mt": eu_mt,
            "EU Obergrenze Mt": gc.constant.get("CO2Limit", float("nan")) / 1e6,
            "DE Emissionen (Budget-Abgrenzung) Mt": de.sum(),
            "Zielfunktion Mrd. EUR": n.objective / 1e9,
            "davon CO2-Zahlung Mrd. EUR": preis * eu_mt / 1e3,
        })

df = pd.DataFrame(rows)
df["DE-Budget KSG Mt"] = df["Jahr"].map(budget)
df["Abstand DE zum Budget Mt"] = df["DE Emissionen (Budget-Abgrenzung) Mt"] - df["DE-Budget KSG Mt"]
df["Systemkosten ohne CO2-Zahlung Mrd. EUR"] = df["Zielfunktion Mrd. EUR"] - df["davon CO2-Zahlung Mrd. EUR"]
print(df.sort_values(["Jahr", "Lauf"]).round(1).T.to_string())

tab = pd.DataFrame(sektoren).fillna(0).round(1)
tab = tab[tab.abs().max(axis=1) > 0.5]
print("\nDE-Emissionen nach Technologie (Mt, Budget-Abgrenzung; + Emission / − Entnahme)")
print(tab[[c for c in tab.columns if c[1] in (2040, 2045)]].to_string())

### Finale Analyse für Präsi:

In [0]:
import pypsa
import pandas as pd
from pathlib import Path

path = "/Volumes/playground/u_luisa_kamp/pypsa-de_data/SR27-3/results/modified-8/"

def laden(path, year):
    return pypsa.Network(str(Path(path) / f"base_s_27__none_{year}.nc"))

# Zuordnung Modell-Carrier -> SR-Sektor (unbekannte landen unter "Sonstige: <carrier>")
CH4_SEKTOR = {
    "OCGT": "Kraftwerke", "CCGT": "Kraftwerke",
    "urban central gas CHP": "Kraftwerke", "urban central gas CHP CC": "Kraftwerke",
    "gas for industry": "Industrie", "gas for industry CC": "Industrie",
    "rural gas boiler": "Haushalte + GHD", "urban decentral gas boiler": "Haushalte + GHD",
    "urban central gas boiler": "Fernwärme",
    "SMR": "Umwandlung (SMR)", "SMR CC": "Umwandlung (SMR)",
    "gas primary": "Aufkommen fossil", "biogas": "Aufkommen Biogas",
    "renewable gas": "Aufkommen EE-Gas Import/Export", "Sabatier": "Aufkommen Sabatier",
    "gas compressing": "Verluste Verdichtung",
}
H2_SEKTOR = {
    "H2 for industry": "Industrie",
    "H2 OCGT": "Kraftwerke", "H2 CCGT": "Kraftwerke",
    "H2 retrofit OCGT": "Kraftwerke", "H2 retrofit CCGT": "Kraftwerke",
    "urban central H2 CHP": "Kraftwerke", "urban central H2 retrofit CHP": "Kraftwerke",
    "H2 Fuel Cell": "Kraftwerke",
    "land transport fuel cell": "Verkehr",
    "methanolisation": "Umwandlung (E-Fuels)", "Fischer-Tropsch": "Umwandlung (E-Fuels)",
    "Sabatier": "Umwandlung (E-Fuels)", "Haber-Bosch": "Umwandlung (E-Fuels)",
    "H2 Electrolysis": "Aufkommen Elektrolyse",
    "SMR": "Aufkommen SMR", "SMR CC": "Aufkommen SMR",
    "H2 pipeline": "Netto-Import Pipeline", "H2 pipeline (Kernnetz)": "Netto-Import Pipeline",
    "H2 pipeline retrofitted": "Netto-Import Pipeline",
    "H2 Store": "Speicher (netto)",
}
SR_CH4 = {2040: {"Kraftwerke": 0, "Industrie": 130, "Haushalte + GHD": 219, "Fernwärme": 15},
          2045: {"Kraftwerke": 56, "Industrie": 101, "Haushalte + GHD": 76, "Fernwärme": 15}}
SR_H2 = {2040: {"Kraftwerke": 20, "Industrie": 79}, 2045: {"Kraftwerke": 27, "Industrie": 101}}


def sektorbilanz(n, bus_filter, mapping):
    """Energiebilanz deutscher Busse in TWh, nach SR-Sektor (− Entnahme, + Aufkommen)."""
    b = n.statistics.energy_balance(groupby=["bus", "carrier"], nice_names=False)
    b = b.droplevel("component")
    bus = b.index.get_level_values("bus")
    b = b[bus.str.startswith("DE") & bus_filter(bus)]
    b = b.groupby(level="carrier").sum() / 1e6
    sektor = b.index.map(lambda c: mapping.get(c, f"Sonstige: {c}"))
    return b.groupby(sektor).sum()


for year in [2040, 2045]:
    n = laden(path, year)
    ch4 = sektorbilanz(n, lambda bus: bus.str.contains("gas"), CH4_SEKTOR)
    h2 = sektorbilanz(n, lambda bus: bus.map(n.buses.carrier) == "H2", H2_SEKTOR)

    for name, bil, sr in [("Methan", ch4, SR_CH4[year]), ("Wasserstoff", h2, SR_H2[year])]:
        df = pd.DataFrame({"Modell TWh": bil.round(1)})
        df["SR S3 TWh (Entnahme)"] = pd.Series(sr).reindex(df.index)
        df = df[df["Modell TWh"].abs() > 0.1]
        print(f"--- {name} DE {year} (− Entnahme / + Aufkommen) ---")
        print(df.sort_values("Modell TWh").to_string())
        print()